# Park route plan prepare

This notebook computes task counts per park from the local task files and visualises the parks with `park_ha_level >= 8` before applying any task filters.

- Parks by task count.


In [ ]:
import pandas as pd
import altair as alt

# Load park table and task files
parks = pd.read_csv("park_table.csv", encoding="utf-8-sig")
task_files = ["park_feature_join_tree_flower.csv", "park_feature_join_artwork_landmark.csv"]
frames = []
for p in task_files:
    frame = pd.read_csv(p, encoding="utf-8-sig")
    frames.append(frame)
tasks = pd.concat(frames, ignore_index=True)

# Compute task counts per park 
counts = tasks.groupby("park_id").size().reset_index(name="task_count")
parks["park_id"] = pd.to_numeric(parks["park_id"], errors="coerce")
counts["park_id"] = pd.to_numeric(counts["park_id"], errors="coerce")
merged = parks.merge(counts, on="park_id", how="left")
merged["task_count"] = merged["task_count"].fillna(0).astype(int)
merged["park_ha_level"] = pd.to_numeric(merged["park_ha_level"], errors="coerce")

# Keep parks with park_ha_level >= 6 only
filtered = merged[merged["park_ha_level"] >= 6].copy()

# Show all parks sorted by task_count
all_parks_sorted = filtered.sort_values("task_count", ascending=False)
bar = alt.Chart(all_parks_sorted).mark_bar().encode(
    x=alt.X("task_count:Q", title="Task count"),
    y=alt.Y("park_name:N", sort="-x", title="Park name"),
    color=alt.Color("park_ha_level:Q", scale=alt.Scale(scheme="viridis")),
    tooltip=["park_name:N", "park_id:Q", "task_count:Q", "park_ha_level:Q"]
).properties(title="Parks by task count before task filtering (park_ha_level >= 6)", height=700)

bar.configure_axis(labelLimit=200)


alt.Chart(...)

In [11]:
# Remove landmark tasks and task_count >= 3
if "feature_source" in tasks.columns:
    tasks = tasks[tasks["feature_source"].astype(str).str.lower() != "landmark"].copy()

# Compute task counts per park after filtering
counts = tasks.groupby("park_id").size().reset_index(name="task_count")
parks["park_id"] = pd.to_numeric(parks["park_id"], errors="coerce")
counts["park_id"] = pd.to_numeric(counts["park_id"], errors="coerce")
merged = parks.merge(counts, on="park_id", how="left")
merged["task_count"] = merged["task_count"].fillna(0).astype(int)
merged["park_ha_level"] = pd.to_numeric(merged["park_ha_level"], errors="coerce")

# Keep parks with park_ha_level >= 6 and at least 3 tasks
filtered = merged[(merged["park_ha_level"] >= 6) & (merged["task_count"] >= 3)].copy()

# Show all parks sorted by task_count
all_parks_sorted = filtered.sort_values("task_count", ascending=False)
bar = alt.Chart(all_parks_sorted).mark_bar().encode(
    x=alt.X("task_count:Q", title="Task count"),
    y=alt.Y("park_name:N", sort="-x", title="Park name"),
    color=alt.Color("park_ha_level:Q", scale=alt.Scale(scheme="viridis")),
    tooltip=["park_name:N", "park_id:Q", "task_count:Q", "park_ha_level:Q"]
).properties(title="Parks by task count after removing landmark tasks and keeping task_count >= 3", height=700)

bar.configure_axis(labelLimit=200)


alt.Chart(...)

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("output")
roads_path = OUTPUT_DIR / "parks_osm_internal_roads.geojson"
entrances_path = OUTPUT_DIR / "parks_osm_gates_entrances.geojson"
boundaries_path = OUTPUT_DIR / "parks_with_boundaries.geojson"

print("OSM API export files expected from: python park_osm_prepare.py")
for path in [boundaries_path, roads_path, entrances_path]:
    print(f"{path}: {'found' if path.exists() else 'missing'}")


In [ ]:
import geopandas as gpd

gdf = gpd.read_file("output/parks_osm_internal_roads.geojson")

print(gdf.columns.tolist())
print(gdf.head(3).T)

In [ ]:
import folium
import geopandas as gpd
from IPython.display import display
from pathlib import Path

OUTPUT_DIR = Path("output")
roads_path = OUTPUT_DIR / "parks_osm_internal_roads.geojson"
entrances_path = OUTPUT_DIR / "parks_osm_gates_entrances.geojson"
boundaries_path = OUTPUT_DIR / "parks_with_boundaries.geojson"

roads = gpd.read_file(roads_path).to_crs("EPSG:4326")
entrances = gpd.read_file(entrances_path).to_crs("EPSG:4326") if entrances_path.exists() else gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
boundaries = gpd.read_file(boundaries_path).to_crs("EPSG:4326") if boundaries_path.exists() else gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

center_geom = roads.to_crs("EPSG:7855").union_all().centroid
center = gpd.GeoSeries([center_geom], crs="EPSG:7855").to_crs("EPSG:4326").iloc[0]
m = folium.Map(location=[center.y, center.x], zoom_start=13, tiles="OpenStreetMap")

if not boundaries.empty:
    boundary_fields = [field for field in ["park_id", "park_name", "park_ha_level"] if field in boundaries.columns]
    folium.GeoJson(
        boundaries[boundary_fields + ["geometry"]],
        name=f"Park boundaries ({len(boundaries)})",
        style_function=lambda _: {"color": "#176f3a", "weight": 2, "fillColor": "#69bd73", "fillOpacity": 0.08},
        tooltip=folium.GeoJsonTooltip(fields=["park_name"], aliases=["Park"]),
    ).add_to(m)

road_colors = {
    "footway": "#1f78ff",
    "path": "#2a9d8f",
    "pedestrian": "#8e44ad",
    "cycleway": "#f4a261",
    "steps": "#e76f51",
    "service": "#6c757d",
    "track": "#795548",
}

roads_layer = folium.FeatureGroup(name=f"OSM internal roads ({len(roads)})", show=True)
for _, row in roads.iterrows():
    road_type = str(row.get("road_type") or "unknown")
    color = road_colors.get(road_type, "#444444")
    popup = folium.Popup(
        "<br>".join([
            f"<b>Park:</b> {row.get('park_name', '')}",
            f"<b>OSM:</b> {row.get('osm_display_id', '')}",
            f"<b>Type:</b> {road_type}",
            f"<b>Surface:</b> {row.get('road_surface', '')}",
            f"<b>Access:</b> {row.get('road_access', '')}",
            f"<b>Name:</b> {row.get('road_name', '')}",
            f"<b>Length m:</b> {row.get('length_m', '')}",
        ]),
        max_width=360,
    )
    folium.GeoJson(
        row.geometry,
        style_function=lambda _, color=color: {"color": color, "weight": 3, "opacity": 0.82},
        tooltip=f"{row.get('park_name', '')}: {road_type}",
        popup=popup,
    ).add_to(roads_layer)
roads_layer.add_to(m)

entrance_layer = folium.FeatureGroup(name=f"OSM gates / entrances ({len(entrances)})", show=True)
for _, row in entrances.iterrows():
    label = row.get("name") or row.get("entrance") or row.get("barrier") or "gate/entrance"
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4,
        color="#ffffff",
        weight=1,
        fill=True,
        fill_color="#e63946",
        fill_opacity=0.95,
        tooltip=f"{row.get('park_name', '')}: {label}",
        popup=folium.Popup(
            f"<b>Park:</b> {row.get('park_name', '')}<br>"
            f"<b>Entrance:</b> {label}<br>"
            f"<b>OSM:</b> {row.get('osm_id', '')}",
            max_width=320,
        ),
    ).add_to(entrance_layer)
entrance_layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
display(m)
